# Python Data & APIs: Data Analysis Application

**Goal of this notebook:** Learn data analysis with **NumPy** and **Pandas**, make charts with **Matplotlib** and **Seaborn**, get data from **REST APIs**, and build a small **Data Analysis Application**.

## What you will learn
1. Data analysis basics with NumPy and Pandas DataFrames
2. Data cleaning and data manipulation
3. Visual plotting with Matplotlib and Seaborn
4. Consuming REST APIs with HTTP requests and processing JSON
5. A complete mini project: load data, clean it, analyze it, and generate a report

## Table of Contents
1. Setup
2. NumPy Basics
3. Pandas Basics
4. Loading the Dataset
5. Data Cleaning
6. Data Manipulation
7. Data Visualization
8. REST APIs and JSON
9. Data Analysis Application (Full Project)
10. Documentation and Conclusion

## 1. Setup

We import the libraries we need. If a library is missing, install it once with:
`pip install numpy pandas matplotlib seaborn requests`

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests

sns.set_theme(style="whitegrid")     # nice default chart style
pd.set_option("display.max_columns", 20)

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

## 2. NumPy Basics

**NumPy** is a library for working with numbers and arrays. It is fast and is the base of Pandas.

- **Array:** a list of numbers with the same type.
- NumPy does math on the **whole array at once** (no loop needed).

In [ ]:
marks = np.array([45, 78, 62, 90, 55, 83])

print("Array     :", marks)
print("Shape     :", marks.shape)
print("Mean      :", marks.mean())
print("Median    :", np.median(marks))
print("Max / Min :", marks.max(), "/", marks.min())
print("Std dev   :", round(marks.std(), 2))

# Math on the whole array at once
print("Marks + 5 :", marks + 5)

# Filtering with a condition (boolean mask)
print("Passed (>=50):", marks[marks >= 50])

In [ ]:
# 2D array (rows and columns)
table = np.array([[1, 2, 3],
                  [4, 5, 6]])

print(table)
print("Shape:", table.shape)
print("Sum of each column:", table.sum(axis=0))   # axis=0 -> down the columns
print("Sum of each row   :", table.sum(axis=1))   # axis=1 -> across the rows

# Handy ways to create arrays
print(np.arange(0, 10, 2))       # 0, 2, 4, 6, 8
print(np.zeros(3))               # three zeros
print(np.linspace(0, 1, 5))      # 5 numbers between 0 and 1

## 3. Pandas Basics

**Pandas** is the main library for data analysis in Python.

| Term | Meaning |
|---|---|
| **Series** | One column of data |
| **DataFrame** | A full table (rows and columns), like an Excel sheet |

In [ ]:
students = pd.DataFrame({
    "name":  ["Ali", "Sara", "Ahmed", "Zoya", "Bilal"],
    "class": ["A", "B", "A", "B", "A"],
    "marks": [78, 92, 65, 88, 54],
})

print(students)
print()
print("Shape:", students.shape)          # (rows, columns)

In [ ]:
# Selecting data
print(students["name"])                   # one column (Series)
print()
print(students[["name", "marks"]])        # many columns
print()
print(students.loc[0])                    # one row by label
print()

# Filtering rows
print(students[students["marks"] > 70])

# Sorting
print(students.sort_values("marks", ascending=False))

# Group by: average marks per class
print(students.groupby("class")["marks"].mean())

### Useful commands to inspect any DataFrame
| Command | What it shows |
|---|---|
| `df.head()` | First 5 rows |
| `df.info()` | Columns, data types, missing values |
| `df.describe()` | Basic statistics (mean, min, max...) |
| `df.isnull().sum()` | Number of missing values in each column |
| `df.duplicated().sum()` | Number of duplicate rows |

## 4. Loading the Dataset

In real projects, data comes from a CSV file. To keep this notebook **self-contained**, we first create a small **messy sales dataset** and save it as `sales_data.csv`. Then we load it with `pd.read_csv()`, exactly like a real file.

The data is messy on purpose, so we can practice cleaning:
- missing values
- duplicate rows
- region names written in different ways (`north`, `NORTH`, ` North`)
- one very big wrong value (outlier)

In [ ]:
# ---- Create a messy sample dataset (run once) ----
rng = np.random.default_rng(42)         # fixed seed: same data every time
n = 200

prices = {"Laptop": 900, "Phone": 600, "Tablet": 400, "Headphones": 80, "Monitor": 250}

raw = pd.DataFrame({
    "order_id": range(1001, 1001 + n),
    "date": pd.to_datetime(rng.choice(pd.date_range("2026-01-01", "2026-06-30"), n)).strftime("%Y-%m-%d"),
    "region": rng.choice(["North", "South", "East", "West"], n),
    "product": rng.choice(list(prices), n),
    "quantity": rng.integers(1, 10, n).astype(float),
    "customer_rating": rng.integers(1, 6, n).astype(float),
})
raw["unit_price"] = (raw["product"].map(prices) * rng.uniform(0.9, 1.1, n)).round(2)

# Make it messy
raw.loc[rng.choice(n, 20, replace=False), "region"] = "north"          # wrong case
raw.loc[rng.choice(n, 15, replace=False), "region"] = " SOUTH "        # extra spaces + case
raw.loc[rng.choice(n, 15, replace=False), "quantity"] = np.nan         # missing
raw.loc[rng.choice(n, 12, replace=False), "unit_price"] = np.nan       # missing
raw.loc[rng.choice(n, 20, replace=False), "customer_rating"] = np.nan  # missing
raw.loc[5, "quantity"] = 999                                           # outlier
raw = pd.concat([raw, raw.sample(8, random_state=1)], ignore_index=True)  # duplicates

raw.to_csv("sales_data.csv", index=False)
print("sales_data.csv saved with", len(raw), "rows")

In [ ]:
# ---- Load the CSV file ----
df = pd.read_csv("sales_data.csv")

df.head()

In [ ]:
df.info()

In [ ]:
print("Missing values in each column:")
print(df.isnull().sum())
print()
print("Duplicate rows:", df.duplicated().sum())
print()
print("Region values:", df["region"].unique())

In [ ]:
df.describe()

**What we found:**
- `quantity`, `unit_price` and `customer_rating` have missing values.
- There are duplicate rows.
- `region` has different spellings of the same name.
- `quantity` has a huge maximum (999), which looks like a mistake (outlier).
- `date` is stored as text, not as a real date.

## 5. Data Cleaning

Clean data gives correct results. We fix the problems one by one.

| Problem | Fix |
|---|---|
| Duplicate rows | `drop_duplicates()` |
| Text problems | `str.strip()` and `str.title()` |
| Wrong data type | `pd.to_datetime()` |
| Missing values | Fill with median (or drop the row) |
| Outliers | Remove using the IQR rule |

In [ ]:
clean = df.copy()      # always keep the original safe
print("Rows at start:", len(clean))

# 1. Remove duplicate rows
clean = clean.drop_duplicates()
print("After removing duplicates:", len(clean))

# 2. Fix text: remove spaces and make first letter capital
clean["region"] = clean["region"].str.strip().str.title()
print("Regions now:", clean["region"].unique())

# 3. Convert date text to real dates
clean["date"] = pd.to_datetime(clean["date"])

In [ ]:
# 4. Handle missing values
#    - unit_price: use the median price of the same product
#    - quantity and rating: use the overall median
clean["unit_price"] = clean["unit_price"].fillna(
    clean.groupby("product")["unit_price"].transform("median"))
clean["quantity"] = clean["quantity"].fillna(clean["quantity"].median())
clean["customer_rating"] = clean["customer_rating"].fillna(clean["customer_rating"].median())

print("Missing values left:")
print(clean.isnull().sum())

In [ ]:
# 5. Remove outliers in quantity using the IQR rule
q1 = clean["quantity"].quantile(0.25)
q3 = clean["quantity"].quantile(0.75)
iqr = q3 - q1
upper_limit = q3 + 1.5 * iqr
print("Upper limit for quantity:", upper_limit)

clean = clean[clean["quantity"] <= upper_limit].copy()
clean["quantity"] = clean["quantity"].astype(int)
clean["customer_rating"] = clean["customer_rating"].astype(int)

print("Rows after cleaning:", len(clean))
clean.head()

## 6. Data Manipulation

Now we create new columns and summarize the data.

In [ ]:
# New columns
clean["revenue"] = clean["quantity"] * clean["unit_price"]
clean["month"] = clean["date"].dt.month_name()
clean["month_number"] = clean["date"].dt.month

clean[["order_id", "product", "quantity", "unit_price", "revenue", "month"]].head()

In [ ]:
# Revenue by product
by_product = clean.groupby("product")["revenue"].sum().sort_values(ascending=False)
print(by_product.round(2))

In [ ]:
# Revenue by region, with several statistics at once
by_region = clean.groupby("region").agg(
    total_revenue=("revenue", "sum"),
    orders=("order_id", "count"),
    avg_rating=("customer_rating", "mean"),
).round(2)

by_region

In [ ]:
# Pivot table: region (rows) x product (columns)
pivot = clean.pivot_table(values="revenue", index="region", columns="product", aggfunc="sum").round(0)
pivot

In [ ]:
# Monthly revenue in the correct calendar order
monthly = (clean.groupby(["month_number", "month"])["revenue"].sum()
                .reset_index().sort_values("month_number"))
monthly

## 7. Data Visualization

Charts make patterns easy to see.

| Library | Use |
|---|---|
| **Matplotlib** | The base plotting library. Full control. |
| **Seaborn** | Built on Matplotlib. Prettier charts with less code. |

In [ ]:
# Matplotlib: bar chart
by_product.plot(kind="bar", color="steelblue", figsize=(8, 4))
plt.title("Total Revenue by Product")
plt.xlabel("Product")
plt.ylabel("Revenue")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# Matplotlib: line chart (trend over time)
plt.figure(figsize=(8, 4))
plt.plot(monthly["month"], monthly["revenue"], marker="o", color="green")
plt.title("Monthly Revenue Trend")
plt.xlabel("Month")
plt.ylabel("Revenue")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Matplotlib: histogram (how values are spread)
plt.figure(figsize=(6, 4))
plt.hist(clean["revenue"], bins=15, color="orange", edgecolor="black")
plt.title("Distribution of Order Revenue")
plt.xlabel("Revenue")
plt.ylabel("Number of orders")
plt.tight_layout()
plt.show()

In [ ]:
# Seaborn: bar chart with regions
plt.figure(figsize=(6, 4))
sns.barplot(data=clean, x="region", y="revenue", estimator="sum", errorbar=None)
plt.title("Total Revenue by Region")
plt.tight_layout()
plt.show()

In [ ]:
# Seaborn: box plot (shows median, spread and outliers)
plt.figure(figsize=(8, 4))
sns.boxplot(data=clean, x="product", y="revenue")
plt.title("Revenue Spread for Each Product")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# Seaborn: heatmap of the pivot table
plt.figure(figsize=(8, 4))
sns.heatmap(pivot, annot=True, fmt=".0f", cmap="YlGnBu")
plt.title("Revenue: Region vs Product")
plt.tight_layout()
plt.show()

In [ ]:
# Seaborn: correlation heatmap (how numbers move together)
numbers = clean[["quantity", "unit_price", "customer_rating", "revenue"]]

plt.figure(figsize=(5, 4))
sns.heatmap(numbers.corr(), annot=True, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlation Between Numbers")
plt.tight_layout()
plt.show()

## 8. REST APIs and JSON

### What is an API?
An **API** (Application Programming Interface) lets one program talk to another program. A **REST API** does this over the internet using **HTTP requests**.

### Main HTTP methods
| Method | Use | Example |
|---|---|---|
| `GET` | Read data | Get a list of users |
| `POST` | Create new data | Add a new post |
| `PUT` | Update data | Change a post |
| `DELETE` | Remove data | Delete a post |

### Common status codes
| Code | Meaning |
|---|---|
| `200` | OK (success) |
| `201` | Created (new data added) |
| `404` | Not found |
| `500` | Server error |

### What is JSON?
**JSON** is the text format APIs use to send data. It looks like a Python dictionary. In Python, `response.json()` turns it into dictionaries and lists.

We use the free practice API **JSONPlaceholder** (`jsonplaceholder.typicode.com`).

In [ ]:
# A safe helper function: it never crashes if the internet is off.
def fetch_json(url, params=None, fallback=None):
    '''GET a URL and return JSON data. Return fallback if something goes wrong.'''
    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()            # raises an error for 404, 500, etc.
        return response.json()
    except requests.exceptions.RequestException as error:
        print("API request failed:", error)
        print("Using sample data instead.")
        return fallback


# Small sample data, used only if the internet is not available
sample_users = [
    {"id": 1, "name": "Leanne Graham", "username": "Bret", "email": "leanne@example.com",
     "address": {"city": "Gwenborough"}, "company": {"name": "Romaguera-Crona"}},
    {"id": 2, "name": "Ervin Howell", "username": "Antonette", "email": "ervin@example.com",
     "address": {"city": "Wisokyburgh"}, "company": {"name": "Deckow-Crist"}},
]

In [ ]:
# 1. GET request
url = "https://jsonplaceholder.typicode.com/users"
users = fetch_json(url, fallback=sample_users)

print("Number of users:", len(users))
print("Type of data   :", type(users))
print()
print("First user (JSON):")
print(json.dumps(users[0], indent=2))

In [ ]:
# 2. Look at the response details (status code, headers)
try:
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    print("Status code :", response.status_code)
    print("Content type:", response.headers.get("Content-Type"))
except requests.exceptions.RequestException as error:
    print("Could not get response details:", error)

In [ ]:
# 3. Turn nested JSON into a DataFrame
users_df = pd.json_normalize(users)          # flattens nested fields like address.city
users_df = users_df[["id", "name", "username", "email", "address.city", "company.name"]]
users_df = users_df.rename(columns={"address.city": "city", "company.name": "company"})

users_df

In [ ]:
# 4. GET with query parameters (filter the data)
posts_url = "https://jsonplaceholder.typicode.com/posts"
sample_posts = [{"userId": 1, "id": 1, "title": "sample post one", "body": "sample text"},
                {"userId": 1, "id": 2, "title": "sample post two", "body": "sample text"}]

posts = fetch_json(posts_url, params={"userId": 1}, fallback=sample_posts)
posts_df = pd.DataFrame(posts)

print("Posts written by user 1:", len(posts_df))
posts_df[["id", "title"]].head()

In [ ]:
# 5. POST request (send new data). JSONPlaceholder pretends to save it.
new_post = {"title": "My first API post", "body": "Learning REST APIs", "userId": 1}

try:
    response = requests.post(posts_url, json=new_post, timeout=10)
    response.raise_for_status()
    print("Status code:", response.status_code)     # 201 means Created
    print("Server reply:", response.json())
except requests.exceptions.RequestException as error:
    print("POST failed:", error)

In [ ]:
# 6. Analyze API data with Pandas: how many posts does each user have?
all_posts = fetch_json(posts_url, fallback=sample_posts)
all_posts_df = pd.DataFrame(all_posts)

posts_per_user = all_posts_df.groupby("userId")["id"].count()
print(posts_per_user)

### API tips
- Always use `timeout=` so your program does not wait forever.
- Always check the status code (`raise_for_status()`).
- Use `try / except` to handle errors.
- Never put passwords or API keys directly in your code.

## 9. Data Analysis Application (Full Project)

Now we combine everything into one clean application. Each function does **one job**.

### Application steps
| Step | Function | Job |
|---|---|---|
| 1 | `load_data()` | Read the CSV file |
| 2 | `clean_data()` | Fix duplicates, text, dates, missing values, outliers |
| 3 | `analyze_data()` | Calculate totals and best performers |
| 4 | `make_charts()` | Create and save charts |
| 5 | `generate_report()` | Write a text report file |
| 6 | `run_application()` | Runs all steps in order |

In [ ]:
def load_data(path):
    '''Step 1: Load the dataset from a CSV file.'''
    data = pd.read_csv(path)
    print(f"Loaded {len(data)} rows and {data.shape[1]} columns from {path}")
    return data


def clean_data(data):
    '''Step 2: Clean the dataset and add useful columns.'''
    data = data.drop_duplicates().copy()
    data["region"] = data["region"].str.strip().str.title()
    data["date"] = pd.to_datetime(data["date"])

    data["unit_price"] = data["unit_price"].fillna(
        data.groupby("product")["unit_price"].transform("median"))
    data["quantity"] = data["quantity"].fillna(data["quantity"].median())
    data["customer_rating"] = data["customer_rating"].fillna(data["customer_rating"].median())

    q1, q3 = data["quantity"].quantile([0.25, 0.75])
    upper_limit = q3 + 1.5 * (q3 - q1)
    data = data[data["quantity"] <= upper_limit].copy()

    data["quantity"] = data["quantity"].astype(int)
    data["customer_rating"] = data["customer_rating"].astype(int)
    data["revenue"] = data["quantity"] * data["unit_price"]
    data["month"] = data["date"].dt.month_name()
    data["month_number"] = data["date"].dt.month

    print(f"Cleaned data: {len(data)} rows remain")
    return data


def analyze_data(data):
    '''Step 3: Calculate the main results and return them in a dictionary.'''
    by_product = data.groupby("product")["revenue"].sum().sort_values(ascending=False)
    by_region = data.groupby("region")["revenue"].sum().sort_values(ascending=False)
    monthly = (data.groupby(["month_number", "month"])["revenue"].sum()
                   .reset_index().sort_values("month_number"))
    rating_by_product = data.groupby("product")["customer_rating"].mean().sort_values(ascending=False)

    return {
        "total_orders": len(data),
        "total_revenue": data["revenue"].sum(),
        "average_order_value": data["revenue"].mean(),
        "average_rating": data["customer_rating"].mean(),
        "by_product": by_product,
        "by_region": by_region,
        "monthly": monthly,
        "rating_by_product": rating_by_product,
        "best_product": by_product.index[0],
        "best_region": by_region.index[0],
        "best_month": monthly.loc[monthly["revenue"].idxmax(), "month"],
    }

In [ ]:
def make_charts(data, results, folder="charts"):
    '''Step 4: Create charts, save them as images, and show them.'''
    os.makedirs(folder, exist_ok=True)

    # Chart 1: revenue by product
    plt.figure(figsize=(7, 4))
    sns.barplot(x=results["by_product"].index, y=results["by_product"].values)
    plt.title("Revenue by Product")
    plt.ylabel("Revenue")
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.savefig(f"{folder}/revenue_by_product.png")
    plt.show()

    # Chart 2: monthly trend
    plt.figure(figsize=(7, 4))
    plt.plot(results["monthly"]["month"], results["monthly"]["revenue"], marker="o")
    plt.title("Monthly Revenue Trend")
    plt.ylabel("Revenue")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(f"{folder}/monthly_trend.png")
    plt.show()

    # Chart 3: region vs product heatmap
    pivot = data.pivot_table(values="revenue", index="region", columns="product", aggfunc="sum")
    plt.figure(figsize=(8, 4))
    sns.heatmap(pivot, annot=True, fmt=".0f", cmap="YlGnBu")
    plt.title("Revenue: Region vs Product")
    plt.tight_layout()
    plt.savefig(f"{folder}/region_product_heatmap.png")
    plt.show()

    print(f"Charts saved in the '{folder}' folder.")


def generate_report(results, path="sales_report.txt"):
    '''Step 5: Write the findings to a text report file.'''
    lines = [
        "=" * 50,
        "SALES DATA ANALYSIS REPORT",
        "=" * 50,
        "",
        "1. OVERVIEW",
        f"   Total orders         : {results['total_orders']}",
        f"   Total revenue        : {results['total_revenue']:,.2f}",
        f"   Average order value  : {results['average_order_value']:,.2f}",
        f"   Average rating (1-5) : {results['average_rating']:.2f}",
        "",
        "2. KEY FINDINGS",
        f"   Best product : {results['best_product']}",
        f"   Best region  : {results['best_region']}",
        f"   Best month   : {results['best_month']}",
        "",
        "3. REVENUE BY PRODUCT",
    ]
    for product, value in results["by_product"].items():
        lines.append(f"   {product:<12} {value:>12,.2f}")

    lines += ["", "4. REVENUE BY REGION"]
    for region, value in results["by_region"].items():
        lines.append(f"   {region:<12} {value:>12,.2f}")

    lines += ["", "5. AVERAGE RATING BY PRODUCT"]
    for product, value in results["rating_by_product"].items():
        lines.append(f"   {product:<12} {value:>6.2f}")

    text = "\n".join(lines)
    with open(path, "w") as file:
        file.write(text)

    print(text)
    print(f"\nReport saved as {path}")

In [ ]:
def run_application(csv_path="sales_data.csv"):
    '''Step 6: Run the full pipeline.'''
    data = load_data(csv_path)
    data = clean_data(data)
    data.to_csv("sales_data_clean.csv", index=False)     # save the clean data
    results = analyze_data(data)
    make_charts(data, results)
    generate_report(results)
    return data, results


final_data, final_results = run_application()

### Check the output files
The application created these files in the same folder as this notebook:
- `sales_data_clean.csv`: cleaned dataset
- `sales_report.txt`: written report
- `charts/`: saved chart images

In [ ]:
for name in ["sales_data.csv", "sales_data_clean.csv", "sales_report.txt"]:
    print(name, "->", "found" if os.path.exists(name) else "missing")
print("Charts:", os.listdir("charts"))

## 10. Documentation and Conclusion

### Application design
| Part | Description |
|---|---|
| **Input** | `sales_data.csv` (messy sales data) |
| **Processing** | Cleaning, new columns, grouping with Pandas and NumPy |
| **Output** | Clean CSV, charts (PNG) and a text report |
| **Design** | One function for each job (clean code) |

### Cleaning decisions
1. Duplicate rows were removed.
2. Text was standardized (`north`, ` SOUTH ` became `North`, `South`).
3. Missing values were filled with the **median**, because the median is not affected by extreme values.
4. Outliers were removed with the **IQR rule**.

### What we learned
- **NumPy** does fast math on arrays.
- **Pandas** DataFrames help us load, clean, filter, group and summarize data.
- **Matplotlib and Seaborn** turn numbers into charts.
- **REST APIs** give us live data using `GET` and `POST` requests, and the answer comes as **JSON**.
- A good data project follows the steps: **load, clean, analyze, visualize, report**.

### Ideas to improve later
- Use real data from Kaggle or a public API
- Export the report as PDF or HTML
- Add more charts and filters
- Build a small dashboard with Streamlit